# FreshBasket Customer Churn Prediction\n## Phase 1: Data Cleaning, Quality Remediation, and Exploratory Data Analysis (EDA)\n\n**Dataset:** FreshBasket Loyalty Churn Dataset (18 Months: Jan 2023 - Jun 2024)\n**Scope:** Phase 1 Only (No feature engineering, no model training)

### 1. Ingestion and Setup\nWe load the Excel workbook FreshBasket_Loyalty_Churn_Dataset.xlsx using header=1 because row 1 contains a banner header.

In [1]:
import os
from pathlib import Path
import pandas as pd
candidates = [
    Path('data/raw/FreshBasket_Loyalty_Churn_Dataset.xlsx'),
    Path('../data/raw/FreshBasket_Loyalty_Churn_Dataset.xlsx'),
    Path('../../data/raw/FreshBasket_Loyalty_Churn_Dataset.xlsx'),
    Path('FreshBasket_Loyalty_Churn_Dataset.xlsx'),
    Path('../FreshBasket_Loyalty_Churn_Dataset.xlsx')
]
excel_path = str(next((c for c in candidates if c.exists()), candidates[0]))

# Load original raw tables
cust_raw = pd.read_excel(excel_path, sheet_name='fb Customer Profile', header=1)
churn_raw = pd.read_excel(excel_path, sheet_name='fb Churn Label', header=1)
act_raw = pd.read_excel(excel_path, sheet_name='fb Monthly Activity', header=1)

print('Raw Shapes:')
print('  fb Customer Profile:', cust_raw.shape)
print('  fb Churn Label:     ', churn_raw.shape)
print('  fb Monthly Activity:', act_raw.shape)


Raw Shapes:
  fb Customer Profile: (2608, 10)
  fb Churn Label:      (2605, 6)
  fb Monthly Activity: (28073, 12)


### 2. Duplicate Detection and Deduplication\nWe identify and resolve exact full-row duplicates across all tables.

In [2]:
# Check duplicates
print('Exact full-row duplicates:')
print('  Customer Profile:', cust_raw.duplicated().sum())
print('  Churn Label:     ', churn_raw.duplicated().sum())
print('  Monthly Activity:', act_raw.duplicated().sum())

# Deduplicate Profile and Churn
cust_clean = cust_raw.drop_duplicates().copy()
churn_clean = churn_raw.drop_duplicates().copy()

print('Cleaned unique customers in Profile:', cust_clean['CUSTOMER_ID'].nunique())
print('Cleaned unique customers in Churn:  ', churn_clean['CUSTOMER_ID'].nunique())

Exact full-row duplicates:
  Customer Profile: 8
  Churn Label:      5
  Monthly Activity: 58
Cleaned unique customers in Profile: 2600
Cleaned unique customers in Churn:   2600


### 3. Resolving Monthly Activity Duplicates and Negative Spend\nWe audit the 58 duplicate rows in b Monthly Activity, including 2 conflicting pairs where one record had positive spend and the duplicate had negative spend.

In [3]:
# Deduplicate exact rows first
act_dedup = act_raw.drop_duplicates().copy()

# Sort so positive spend takes precedence, then drop customer-month duplicates
act_clean = act_dedup.sort_values(by=['CUSTOMER_ID', 'MONTH', 'TOTAL_SPEND'], ascending=[True, True, False])
act_clean = act_clean.drop_duplicates(subset=['CUSTOMER_ID', 'MONTH'], keep='first').copy()

# Flag negative spend and zero spend with transactions
act_clean['IS_NEGATIVE_SPEND'] = (act_clean['TOTAL_SPEND'] < 0).astype(int)
act_clean['IS_ZERO_SPEND_WITH_TXNS'] = ((act_clean['TRANSACTIONS'] > 0) & (act_clean['TOTAL_SPEND'] == 0)).astype(int)

# Create reconstructed spend column while preserving raw TOTAL_SPEND
act_clean['TOTAL_SPEND_RECONSTRUCTED'] = act_clean['TOTAL_SPEND']
act_clean.loc[act_clean['IS_NEGATIVE_SPEND'] == 1, 'TOTAL_SPEND_RECONSTRUCTED'] = act_clean.loc[act_clean['IS_NEGATIVE_SPEND'] == 1, 'TOTAL_SPEND'].abs()
act_clean.loc[act_clean['IS_ZERO_SPEND_WITH_TXNS'] == 1, 'TOTAL_SPEND_RECONSTRUCTED'] = (act_clean.loc[act_clean['IS_ZERO_SPEND_WITH_TXNS'] == 1, 'TRANSACTIONS'] * act_clean.loc[act_clean['IS_ZERO_SPEND_WITH_TXNS'] == 1, 'AVG_BASKET_VALUE']).round(2)

print('Cleaned Activity Shape:', act_clean.shape)
print('Remaining Negative Spend records flagged:', act_clean['IS_NEGATIVE_SPEND'].sum())
print('Zero Spend with Transactions flagged:   ', act_clean['IS_ZERO_SPEND_WITH_TXNS'].sum())

Cleaned Activity Shape: (28013, 15)
Remaining Negative Spend records flagged: 22
Zero Spend with Transactions flagged:    20


### 4. Categorical Standardization and Missing Value Handling\nWe standardize whitespace/casing in CITY and MEMBERSHIP_TIER, and handle missing values cleanly.

In [4]:
# Standardize text
cust_clean['CITY_RAW'] = cust_clean['CITY']
cust_clean['CITY'] = cust_clean['CITY'].astype(str).str.strip().str.title()

cust_clean['MEMBERSHIP_TIER_RAW'] = cust_clean['MEMBERSHIP_TIER']
cust_clean['MEMBERSHIP_TIER'] = cust_clean['MEMBERSHIP_TIER'].astype(str).str.strip().str.title()

# Clean missing categoricals and flag missing numericals
cust_clean['GENDER_CLEANED'] = cust_clean['GENDER'].fillna('Unknown')
cust_clean['HOME_STORE_PRICE_TIER_CLEANED'] = cust_clean['HOME_STORE_PRICE_TIER'].fillna('Unknown')
cust_clean['AGE_IS_MISSING'] = cust_clean['AGE'].isnull().astype(int)

print('Unique Cities:', cust_clean['CITY'].nunique())
print('Unique Tiers: ', cust_clean['MEMBERSHIP_TIER'].unique())
print('Missing Age count:', cust_clean['AGE_IS_MISSING'].sum())

Unique Cities: 10
Unique Tiers:  ['Silver' 'Gold' 'Platinum']
Missing Age count: 78


### 5. Exporting Cleaned Data Tables\nSave clean CSV outputs in phase_1_outputs/cleaned_data/.

In [5]:
from pathlib import Path
root_candidates = [Path('.'), Path('..'), Path('../..')]
root = next((p for p in root_candidates if (p / 'phases').exists() and (p / 'data').exists()), Path('.'))
out_dir = root / 'phases' / 'phase1' / 'output' / 'cleaned_data'
out_dir.mkdir(parents=True, exist_ok=True)

cust_clean.to_csv(out_dir / 'cleaned_fb_customer_profile.csv', index=False)
churn_clean.to_csv(out_dir / 'cleaned_fb_churn_label.csv', index=False)
act_clean.to_csv(out_dir / 'cleaned_fb_monthly_activity.csv', index=False)
print(f'Cleaned CSV files successfully written to {out_dir}.')


Cleaned CSV files successfully written to ..\..\phases\phase1\output\cleaned_data.


### 6. Verification of the Churn Target\nVerify the official churn definition: zero transactions in April-June 2024 despite prior purchase history.

In [6]:
# Target window transactions
target_months = ['2024-04-01', '2024-05-01', '2024-06-01']
act_target = act_clean[act_clean['MONTH'].astype(str).str[:10].isin(target_months)]
target_txns = act_target.groupby('CUSTOMER_ID')['TRANSACTIONS'].sum().reset_index()

m_churn = churn_clean.merge(target_txns, on='CUSTOMER_ID', how='left')
m_churn['TRANSACTIONS'] = m_churn['TRANSACTIONS'].fillna(0)

print('Target verification table:')
print(pd.crosstab(m_churn['TRANSACTIONS'] > 0, m_churn['CHURNED'], rownames=['Target_Txns > 0'], colnames=['CHURNED']))

Target verification table:
CHURNED             0    1
Target_Txns > 0           
False             125  818
True             1657    0


### 7. Exploratory Data Analysis Summaries\nExamine demographic distributions, temporal trends, and pre-cutoff behavioral patterns.

In [7]:
# Merge cleaned profile and churn
df_cust = cust_clean.merge(churn_clean, on='CUSTOMER_ID')

# Pre-cutoff behavioral aggregation (Jan 2023 - Mar 2024)
act_pre = act_clean[act_clean['MONTH'] < '2024-04-01']
pre_summary = act_pre.groupby('CUSTOMER_ID').agg(
    monthly_spend=('TOTAL_SPEND_RECONSTRUCTED', 'mean'),
    monthly_txns=('TRANSACTIONS', 'mean'),
    app_sessions=('APP_SESSIONS', 'mean'),
    complaint_flag=('COMPLAINT_FLAG', 'max')
).reset_index()

df_eda = df_cust.merge(pre_summary, on='CUSTOMER_ID', how='left')

print('Churn Rate by Membership Tier:')
print(df_eda.groupby('MEMBERSHIP_TIER')['CHURNED'].mean() * 100)

print('\nBehavioral Differences (Pre-Cutoff):')
print(df_eda.groupby('CHURNED')[['monthly_spend', 'monthly_txns', 'app_sessions', 'complaint_flag']].mean())

Churn Rate by Membership Tier:
MEMBERSHIP_TIER
Gold        30.097087
Platinum    31.125828
Silver      32.293080
Name: CHURNED, dtype: float64

Behavioral Differences (Pre-Cutoff):
         monthly_spend  monthly_txns  app_sessions  complaint_flag
CHURNED                                                           
0           140.146004      4.332265      2.144629        0.306907
1            75.482853      2.360425      1.077916        0.382353
